In [5]:
#搭建并训练一个简单CNN，用于图像分类
from collections.abc import Callable

import dnnlpy
import dnnlpy.nn as dnn
import dnnlpy.optim as dopt
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as utils
import torchinfo
import torchmetrics.classification as metrics
import torchvision.datasets as datasets
import torchvision.transforms.v2 as v2
from torch import Tensor

dnnlpy.set_matplotlib_format('svg')
print('PyTorch version:', torch.__version__)

PyTorch version: 2.14.0


In [6]:
#基础卷积块：Conv2d → ReLU → MaxPool2d
block=nn.Sequential(
    dnn.Conv2d(3,16,kernel_size=3,padding=1),
    dnn.ReLU(),
    dnn.MaxPool2d(kernel_size=2),
)

x=torch.randn(16,3,28,28)
y=block(x)

print('Input shape:',x.shape)
print('Output shape:',y.shape)

Input shape: torch.Size([16, 3, 28, 28])
Output shape: torch.Size([16, 16, 14, 14])


In [7]:
#CNN 通常随着网络加深逐步输出数据的空间尺寸减少，同时增加通道数/特征图数
#两个卷积块
features=nn.Sequential(
    dnn.Conv2d(3,16,kernel_size=3,padding=1),
    dnn.ReLU(),
    dnn.MaxPool2d(kernel_size=2,stride=2),
    dnn.Conv2d(16,32,kernel_size=3,padding=1),
    dnn.ReLU(),
    dnn.MaxPool2d(kernel_size=2,stride=2),
)

x=torch.randn(16,3,28,28)
y=features(x)

print('Input shape:',x.shape)
print('Output shape:',y.shape)
#输出结果看，通道数/特征图数增加，空间尺寸减少

Input shape: torch.Size([16, 3, 28, 28])
Output shape: torch.Size([16, 32, 7, 7])


In [8]:
#定义SimpleCNN
class SimpleCNN(nn.Module): #继承神经网络模块的基类nn.Module
    def __init__(self,in_channels:int=1,num_classes:int=3,)->None:
        super().__init__() #调用父类nn.Module去初始化
        self.num_classes=num_classes
        self.features=nn.Sequential(
            dnn.Conv2d(in_channels,16,kernel_size=3,padding=1),
            dnn.ReLU(),
            dnn.MaxPool2d(kernel_size=2),
            dnn.Conv2d(16,32,kernel_size=3,padding=1),
            dnn.ReLU(),
            dnn.MaxPool2d(kernel_size=2),
        )
        self.pool=dnn.AdaptiveAvgPool2d(1)
        self.flatten=dnn.Flatten()
        self.classifier=dnn.Linear(32,num_classes)

    def forward(self,x:Tensor)->Tensor:
        x=self.features(x)
        x=self.pool(x)
        x=self.flatten(x)
        x=self.classifier(x)
        return x

In [10]:
#将SimpleCNN实例化为model，并逐层输出形状
model=SimpleCNN(in_channels=3,num_classes=3)

x=torch.randn(16,3,28,28)
logits=model(x)

print(model)
print('Input shape:',x.shape)
print('Logits shape:',logits.shape)

summary = torchinfo.summary(model, input_size=(16, 3, 28, 28))
print(summary)

SimpleCNN(
  (features): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1, 1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1, 1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (pool): AdaptiveAvgPool2d(output_size=1)
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (classifier): Linear(in_features=32, out_features=3, bias=True)
)
Input shape: torch.Size([16, 3, 28, 28])
Logits shape: torch.Size([16, 3])
Layer (type:depth-idx)                   Output Shape              Param #
SimpleCNN                                [16, 3]                   --
├─Sequential: 1-1                        [16, 32, 7, 7]            --
│    └─Conv2d: 2-1                       [16, 16, 28, 28]          448
│    └─ReLU: 2-2                         [16, 16, 28, 28]          --
│    └─Max